# AI Programming — Lecture 5
## Neural Networks: Forward Pass와 Backpropagation을 쉽게 이해하기

이 노트북은 Lecture 5의 **Perceptron, MLP, Forward Pass, Backward Pass**를
수식만으로 따라가기보다 **작은 숫자를 직접 계산하면서 이해**하기 위한 보조 실습입니다.

특히 backpropagation에서는 처음부터 큰 행렬과 Jacobian을 다루지 않습니다.

> **핵심 아이디어**
>
> 1. Forward pass: 값을 앞쪽으로 계산한다.
> 2. Loss: 예측이 얼마나 틀렸는지 계산한다.
> 3. Backward pass: 각 값이 loss에 얼마나 영향을 주었는지 뒤쪽으로 계산한다.
> 4. Gradient descent: loss를 줄이는 방향으로 parameter를 조금 움직인다.

### 학습 목표

실습을 마치면 다음 내용을 설명할 수 있어야 합니다.

- 하나의 neuron이 `weighted sum → activation`으로 동작함을 이해합니다.
- 여러 neuron을 연결하면 MLP가 된다는 것을 이해합니다.
- Forward pass에서 중간값을 저장하는 이유를 설명할 수 있습니다.
- gradient를 **"parameter를 조금 바꾸었을 때 loss가 얼마나 변하는가"**로 이해합니다.
- chain rule을 **upstream gradient × local gradient**로 이해합니다.
- 하나의 neuron에 대해 backpropagation을 직접 계산할 수 있습니다.
- 2-layer network에서 gradient가 뒤쪽 layer에서 앞쪽 layer로 전달되는 과정을 이해합니다.
- TensorFlow의 automatic differentiation이 같은 gradient를 계산함을 확인합니다.
- 깊은 network에서 gradient가 작아질 수 있는 이유를 직관적으로 이해합니다.

### 실습 방법

1. 셀을 위에서부터 순서대로 실행하세요.
2. 계산 결과를 외우기보다 **중간값이 어디에서 왔는지** 확인하세요.
3. `TODO`가 표시된 부분은 값을 직접 바꾸어 다시 실행하세요.
4. Backpropagation에서 가장 중요한 것은 긴 식이 아니라 **계산 방향**입니다.

**Forward:** 입력 → 예측  
**Backward:** Loss → parameter

## 0. 라이브러리 불러오기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)

# Part I. Forward Pass

## 1. 하나의 Neuron

하나의 neuron은 먼저 weighted sum을 계산합니다.

$$
z = w_1x_1 + w_2x_2 + b
$$

그 다음 activation function을 적용합니다.

$$
h = \sigma(z)
$$

따라서 전체 계산은

$$
h = \sigma(\mathbf{w}^{\top}\mathbf{x}+b)
$$

입니다.

처음에는 이 식을 한 번에 보지 말고 두 단계로 나누어 생각하는 것이 좋습니다.

```text
x  ── weighted sum ──> z ── activation ──> h
```

In [ ]:
x = np.array([2.0, 1.0])
w = np.array([0.5, -1.0])
b = 0.2

# Step 1. weighted sum
z = w @ x + b

# Step 2. activation
h = max(0.0, z)   # ReLU

print("x =", x)
print("w =", w)
print("b =", b)
print("z = w @ x + b =", z)
print("h = ReLU(z) =", h)

### 확인할 내용

`z`와 `h`를 구분해서 보세요.

- `z`: activation을 통과하기 전 값 (**pre-activation**)
- `h`: activation을 통과한 값 (**activation/output**)

Backpropagation에서도 이 두 값을 구분하는 것이 매우 중요합니다.

## 2. Activation Function

Lecture 5에서는 sigmoid와 ReLU가 등장합니다.

### Sigmoid

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

### ReLU

$$
\mathrm{ReLU}(z)=\max(0,z)
$$

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def relu(z):
    return np.maximum(0.0, z)

z = np.linspace(-6, 6, 300)

plt.plot(z, sigmoid(z), label="Sigmoid")
plt.plot(z, relu(z), label="ReLU")
plt.axvline(0, linestyle="--")
plt.xlabel("z")
plt.ylabel("activation")
plt.title("Activation Functions")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. Neuron을 여러 개 연결하면 MLP

2-layer MLP를 아주 작게 생각해 봅니다.

```text
input x
  ↓
hidden layer
  ↓
output layer
  ↓
prediction
```

수식으로는

$$
\mathbf{h}
=
\sigma_h(\mathbf{W}^{(1)}\mathbf{x}+\mathbf{b}^{(1)})
$$

$$
\hat{\mathbf{y}}
=
\sigma_y(\mathbf{W}^{(2)}\mathbf{h}+\mathbf{b}^{(2)})
$$

입니다.

중요한 점은 **두 개의 함수가 순서대로 연결된 composite function**이라는 것입니다.

In [ ]:
x = np.array([1.0, 2.0])

W1 = np.array([
    [0.5, 0.2],
    [-0.3, 0.8]
])
b1 = np.array([0.1, -0.1])

W2 = np.array([
    [0.7, -0.4]
])
b2 = np.array([0.2])

# Hidden layer
z1 = W1 @ x + b1
h = relu(z1)

# Output layer
z2 = W2 @ h + b2
y_hat = z2.copy()   # regression을 가정하여 output activation 생략

print("x shape   :", x.shape)
print("W1 shape  :", W1.shape)
print("z1 shape  :", z1.shape)
print("h shape   :", h.shape)
print("W2 shape  :", W2.shape)
print("y_hat shape:", y_hat.shape)

print("\nz1 =", z1)
print("h  =", h)
print("y_hat =", y_hat)

> ### ✅ 체크포인트
>
> Forward pass에서는 **값(value)**을 계산합니다.
>
> ```text
> x → z1 → h → z2 → y_hat
> ```
>
> 이 중간값들은 backward pass에서 다시 사용됩니다.

## 4. Hidden Layer는 새로운 표현을 만든다: XOR

하나의 perceptron은 하나의 linear decision boundary만 만들 수 있기 때문에 XOR를 직접 분리할 수 없습니다.

Hidden layer를 사용하면 입력 $\mathbf{x}$를 새로운 표현 $\mathbf{h}$로 바꾼 뒤,
그 표현 공간에서 문제를 더 쉽게 풀 수 있습니다.

이번 셀에서는 학습 자체보다 **hidden representation**의 의미에 집중합니다.

In [ ]:
X_xor = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

y_xor = np.array([0, 1, 1, 0])

plt.scatter(
    X_xor[:, 0],
    X_xor[:, 1],
    c=y_xor
)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("XOR in Input Space")
plt.xticks([0, 1])
plt.yticks([0, 1])
plt.grid(alpha=0.3)
plt.show()

XOR의 핵심 메시지는 다음 한 문장으로 충분합니다.

> **Hidden layer는 입력을 새로운 representation으로 바꾸어, 원래 공간에서 어려웠던 문제를 더 쉽게 만들 수 있다.**

# Part II. Backward Pass를 위한 준비

## 5. Loss는 "얼마나 틀렸는가"

간단한 regression 예제로 생각해 봅니다.

예측값이 $\hat{y}$, 정답이 $y$라면

$$
\mathcal{L}
=
\frac{1}{2}(\hat{y}-y)^2
$$

를 사용하겠습니다.

앞에 $\frac{1}{2}$를 붙이는 이유는 미분할 때 2가 없어져 계산이 간단해지기 때문입니다.

In [ ]:
y_hat = 3.0
y = 5.0

loss = 0.5 * (y_hat - y) ** 2

print("prediction =", y_hat)
print("target     =", y)
print("loss       =", loss)

## 6. Gradient를 먼저 직관적으로 이해하기

Gradient는 어렵게 생각할 필요가 없습니다.

> **Parameter를 아주 조금 바꾸었을 때 loss가 얼마나 변하는가?**

예를 들어 parameter $w$가 있고 loss가 $\mathcal{L}(w)$라면

$$
\frac{\partial \mathcal{L}}{\partial w}
$$

는 $w$를 조금 증가시켰을 때 loss가 어느 방향으로 얼마나 변하는지를 나타냅니다.

- gradient > 0 → $w$를 증가시키면 loss가 증가하는 방향
- gradient < 0 → $w$를 증가시키면 loss가 감소하는 방향
- gradient ≈ 0 → $w$를 조금 바꾸어도 loss 변화가 작음

## 7. 미분 없이 Gradient를 추정해 보기

먼저 derivative 공식을 사용하지 않고,
parameter를 아주 조금 움직여 loss 변화를 직접 측정해 봅니다.

이 방법을 **numerical gradient** 또는 **finite difference**라고 합니다.

$$
\frac{\partial \mathcal{L}}{\partial w}
\approx
\frac{
\mathcal{L}(w+\epsilon)
-
\mathcal{L}(w-\epsilon)
}{
2\epsilon
}
$$

In [ ]:
x = 2.0
y = 5.0
b = 1.0

def simple_loss(w):
    y_hat = w * x + b
    return 0.5 * (y_hat - y) ** 2

w = 1.0
eps = 1e-5

loss_plus = simple_loss(w + eps)
loss_minus = simple_loss(w - eps)

grad_numerical = (
    loss_plus - loss_minus
) / (2 * eps)

print("loss(w) =", simple_loss(w))
print("numerical gradient =", grad_numerical)

### 확인할 내용

여기서는 backpropagation을 아직 사용하지 않았습니다.

단순히

```text
w를 아주 조금 증가시켜 보고
w를 아주 조금 감소시켜 본 뒤
loss 변화량을 측정
```

했습니다.

즉, gradient의 의미부터 확인한 것입니다.

# Part III. Chain Rule을 한 단계씩 보기

## 8. 가장 작은 Backpropagation 예제

다음 하나의 neuron을 생각합니다.

$$
z = wx+b
$$

$$
\hat{y}=z
$$

$$
\mathcal{L}
=
\frac{1}{2}(\hat{y}-y)^2
$$

Forward pass는

```text
w → z → y_hat → loss
```

입니다.

Backward pass에서는 반대 방향으로

```text
loss → y_hat → z → w
```

를 따라갑니다.

### Chain Rule

우리가 원하는 것은

$$
\frac{\partial \mathcal{L}}{\partial w}
$$

입니다.

하지만 loss는 $w$와 직접 연결되어 있지 않고 중간에 $z$가 있습니다.

따라서

$$
\frac{\partial \mathcal{L}}{\partial w}
=
\frac{\partial \mathcal{L}}{\partial \hat{y}}
\frac{\partial \hat{y}}{\partial z}
\frac{\partial z}{\partial w}
$$

가 됩니다.

이를 외우기보다 다음처럼 읽으면 됩니다.

> **앞에서 전달되어 온 gradient × 지금 연산의 local gradient**

이 값이 다시 이전 단계로 전달됩니다.

In [ ]:
# Forward
x = 2.0
w = 1.0
b = 1.0
y = 5.0

z = w * x + b
y_hat = z
loss = 0.5 * (y_hat - y) ** 2

print("=== Forward ===")
print("z     =", z)
print("y_hat =", y_hat)
print("loss  =", loss)

# Backward
dL_dyhat = y_hat - y     # dL / dy_hat
dyhat_dz = 1.0           # dy_hat / dz
dz_dw = x                 # dz / dw
dz_db = 1.0               # dz / db

dL_dw = dL_dyhat * dyhat_dz * dz_dw
dL_db = dL_dyhat * dyhat_dz * dz_db

print("\n=== Backward ===")
print("dL/dy_hat =", dL_dyhat)
print("dy_hat/dz =", dyhat_dz)
print("dz/dw     =", dz_dw)
print("dL/dw     =", dL_dw)
print("dL/db     =", dL_db)

> ### ✅ 체크포인트
>
> Numerical gradient와 chain rule로 계산한 `dL/dw`가 같은지 비교하세요.
>
> 이 작은 예제가 바로 backpropagation의 핵심입니다.
>
> Network가 커져도 **같은 원리를 여러 번 반복**합니다.

## 9. Gradient Descent로 한 번 Update

Gradient를 계산한 이유는 parameter를 업데이트하기 위해서입니다.

$$
w
\leftarrow
w-\eta\frac{\partial \mathcal{L}}{\partial w}
$$

$$
b
\leftarrow
b-\eta\frac{\partial \mathcal{L}}{\partial b}
$$

$\eta$는 learning rate입니다.

In [ ]:
lr = 0.1

print("업데이트 전")
print("w =", w, "b =", b, "loss =", loss)

w_new = w - lr * dL_dw
b_new = b - lr * dL_db

z_new = w_new * x + b_new
y_hat_new = z_new
loss_new = 0.5 * (y_hat_new - y) ** 2

print("\n업데이트 후")
print("w =", w_new, "b =", b_new, "loss =", loss_new)

### 확인할 내용

Gradient 방향의 **반대 방향**으로 parameter를 움직였더니 loss가 줄었는지 확인하세요.

이것이 training의 한 step입니다.

## 10. Activation Function이 들어가면 한 단계만 더 곱한다

이번에는 sigmoid neuron을 사용합니다.

$$
z = wx+b
$$

$$
\hat{y}=\sigma(z)
$$

$$
\mathcal{L}
=
\frac{1}{2}(\hat{y}-y)^2
$$

이 경우에는 sigmoid의 derivative가 한 단계 더 필요합니다.

$$
\frac{d\sigma(z)}{dz}
=
\sigma(z)(1-\sigma(z))
$$

따라서

$$
\frac{\partial \mathcal{L}}{\partial w}
=
\frac{\partial \mathcal{L}}{\partial \hat{y}}
\frac{\partial \hat{y}}{\partial z}
\frac{\partial z}{\partial w}
$$

입니다.

In [ ]:
x = 2.0
w = 0.5
b = 0.0
y = 1.0

# ----------------
# Forward
# ----------------
z = w * x + b
y_hat = sigmoid(z)
loss = 0.5 * (y_hat - y) ** 2

# ----------------
# Backward
# ----------------
dL_dyhat = y_hat - y
dyhat_dz = y_hat * (1.0 - y_hat)
dz_dw = x
dz_db = 1.0

dL_dz = dL_dyhat * dyhat_dz

dL_dw = dL_dz * dz_dw
dL_db = dL_dz * dz_db

print("Forward")
print("z =", z)
print("y_hat =", y_hat)
print("loss =", loss)

print("\nBackward")
print("dL/dy_hat =", dL_dyhat)
print("dy_hat/dz =", dyhat_dz)
print("dL/dz =", dL_dz)
print("dL/dw =", dL_dw)
print("dL/db =", dL_db)

### 중요하게 볼 것

`dL/dz`를 따로 저장했습니다.

```text
loss
 ↓
dL/dy_hat
 ↓  × sigmoid의 local derivative
dL/dz
 ↓  × weighted sum의 local derivative
dL/dw, dL/db
```

실제 neural network의 backpropagation도 이런 방식으로 **중간 gradient를 하나씩 전달**합니다.

## 11. 우리가 계산한 Gradient가 맞는지 확인하기

Backpropagation으로 구한 gradient와 numerical gradient를 비교합니다.

두 값이 거의 같다면 chain rule 계산이 맞다는 뜻입니다.

In [ ]:
def sigmoid_neuron_loss(w_value):
    z_value = w_value * x + b
    y_hat_value = sigmoid(z_value)
    return 0.5 * (y_hat_value - y) ** 2

eps = 1e-5

grad_numerical = (
    sigmoid_neuron_loss(w + eps)
    - sigmoid_neuron_loss(w - eps)
) / (2 * eps)

print("Backprop gradient :", dL_dw)
print("Numerical gradient:", grad_numerical)
print("Difference        :", abs(dL_dw - grad_numerical))

# Part IV. 2-Layer Network Backpropagation

## 12. 먼저 Scalar Network로 생각하기

행렬을 잠시 내려놓고 다음처럼 숫자 하나씩 연결된 network를 생각합니다.

```text
x
↓
z1 = w1*x + b1
↓
h = sigmoid(z1)
↓
z2 = w2*h + b2
↓
y_hat = z2
↓
loss
```

Forward에서는 위에서 아래로 값을 계산합니다.

Backward에서는 정확히 반대로 gradient를 전달합니다.

```text
loss
↑
y_hat
↑
z2
↑
h
↑
z1
↑
w1
```

In [ ]:
# ----------------
# Parameters
# ----------------
x = 1.5
y = 2.0

w1 = 0.8
b1 = 0.1

w2 = 1.2
b2 = -0.2

# ----------------
# Forward
# ----------------
z1 = w1 * x + b1
h = sigmoid(z1)

z2 = w2 * h + b2
y_hat = z2

loss = 0.5 * (y_hat - y) ** 2

print("z1 =", z1)
print("h  =", h)
print("z2 =", z2)
print("y_hat =", y_hat)
print("loss =", loss)

## 13. Output Layer부터 Backward

먼저 output 쪽 parameter $w_2$를 봅니다.

$$
\frac{\partial \mathcal{L}}{\partial w_2}
=
\frac{\partial \mathcal{L}}{\partial \hat{y}}
\frac{\partial \hat{y}}{\partial z_2}
\frac{\partial z_2}{\partial w_2}
$$

여기서

$$
\frac{\partial z_2}{\partial w_2}=h
$$

입니다.

그리고 hidden layer로 gradient를 보내려면

$$
\frac{\partial \mathcal{L}}{\partial h}
=
\frac{\partial \mathcal{L}}{\partial z_2}
\frac{\partial z_2}{\partial h}
$$

를 계산합니다.

In [ ]:
# Output layer backward
dL_dyhat = y_hat - y

dyhat_dz2 = 1.0
dL_dz2 = dL_dyhat * dyhat_dz2

dz2_dw2 = h
dz2_db2 = 1.0
dz2_dh = w2

dL_dw2 = dL_dz2 * dz2_dw2
dL_db2 = dL_dz2 * dz2_db2

# hidden layer로 전달되는 upstream gradient
dL_dh = dL_dz2 * dz2_dh

print("dL/dz2 =", dL_dz2)
print("dL/dw2 =", dL_dw2)
print("dL/db2 =", dL_db2)
print("dL/dh  =", dL_dh)

## 14. Hidden Layer로 계속 Backward

이제 앞에서 받은 `dL/dh`를 사용합니다.

$$
\frac{\partial \mathcal{L}}{\partial z_1}
=
\frac{\partial \mathcal{L}}{\partial h}
\frac{\partial h}{\partial z_1}
$$

그리고

$$
\frac{\partial \mathcal{L}}{\partial w_1}
=
\frac{\partial \mathcal{L}}{\partial z_1}
\frac{\partial z_1}{\partial w_1}
$$

입니다.

즉 **뒤 layer에서 받은 gradient를 local derivative와 곱해 앞쪽으로 전달**합니다.

In [ ]:
# sigmoid local derivative
dh_dz1 = h * (1.0 - h)

# pre-activation gradient
dL_dz1 = dL_dh * dh_dz1

# parameter gradients
dz1_dw1 = x
dz1_db1 = 1.0

dL_dw1 = dL_dz1 * dz1_dw1
dL_db1 = dL_dz1 * dz1_db1

print("dL/dh  =", dL_dh)
print("dh/dz1 =", dh_dz1)
print("dL/dz1 =", dL_dz1)
print("dL/dw1 =", dL_dw1)
print("dL/db1 =", dL_db1)

> ### ✅ 핵심 정리
>
> ```text
> Forward:   x → z1 → h → z2 → y_hat → loss
>
> Backward:  loss → z2 → h → z1 → parameters
> ```
>
> Backpropagation은 **gradient를 뒤에서 앞으로 전달하는 알고리즘**입니다.
>
> 각 단계에서 하는 일은 하나입니다.
>
> **upstream gradient × local derivative**

## 15. 모든 Parameter는 Gradient 계산 후 함께 Update

Backpropagation을 끝내면 각 parameter에 대한 gradient가 준비됩니다.

그 다음 gradient descent로 parameter들을 업데이트합니다.

In [ ]:
lr = 0.1

print("Before:")
print("w1, b1, w2, b2 =", w1, b1, w2, b2)
print("loss =", loss)

# 모든 gradient가 계산된 뒤 update
w1 = w1 - lr * dL_dw1
b1 = b1 - lr * dL_db1
w2 = w2 - lr * dL_dw2
b2 = b2 - lr * dL_db2

# 다시 forward하여 loss 확인
z1_new = w1 * x + b1
h_new = sigmoid(z1_new)

z2_new = w2 * h_new + b2
y_hat_new = z2_new

loss_new = 0.5 * (y_hat_new - y) ** 2

print("\nAfter:")
print("w1, b1, w2, b2 =", w1, b1, w2, b2)
print("loss =", loss_new)

## 16. 같은 과정을 여러 번 반복하면 Training

Training loop는 사실 다음 네 단계를 반복하는 것입니다.

```text
1. Forward
2. Loss
3. Backward
4. Update
```

아래 코드는 방금 손으로 계산한 과정을 반복합니다.

In [ ]:
# 간단한 1-sample training example

x = 1.5
y = 2.0

w1 = 0.8
b1 = 0.1
w2 = 1.2
b2 = -0.2

lr = 0.1
loss_history = []

for step in range(100):
    # ----------------
    # 1. Forward
    # ----------------
    z1 = w1 * x + b1
    h = sigmoid(z1)

    z2 = w2 * h + b2
    y_hat = z2

    # ----------------
    # 2. Loss
    # ----------------
    loss = 0.5 * (y_hat - y) ** 2
    loss_history.append(loss)

    # ----------------
    # 3. Backward
    # ----------------
    dL_dz2 = y_hat - y

    dL_dw2 = dL_dz2 * h
    dL_db2 = dL_dz2

    dL_dh = dL_dz2 * w2

    dh_dz1 = h * (1 - h)
    dL_dz1 = dL_dh * dh_dz1

    dL_dw1 = dL_dz1 * x
    dL_db1 = dL_dz1

    # ----------------
    # 4. Update
    # ----------------
    w1 -= lr * dL_dw1
    b1 -= lr * dL_db1

    w2 -= lr * dL_dw2
    b2 -= lr * dL_db2

plt.plot(loss_history)
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("Manual Backpropagation")
plt.grid(alpha=0.3)
plt.show()

print("final prediction =", y_hat)
print("target =", y)
print("final loss =", loss_history[-1])

# Part V. Scalar에서 Vector-Matrix Form으로

## 17. 실제 Neural Network에서는 같은 계산을 동시에 수행한다

실제 network에서는 neuron이 여러 개이므로

$$
\mathbf{z}
=
\mathbf{W}\mathbf{x}+\mathbf{b}
$$

와 같이 matrix 연산을 사용합니다.

하지만 개념은 scalar 예제와 같습니다.

```text
Scalar:
z = wx + b

Vector:
z = W @ x + b
```

Backward도 마찬가지로 여러 숫자의 gradient를 한 번에 계산합니다.

여기서부터는 모든 Jacobian을 손으로 전개하기보다
**shape와 gradient의 흐름**을 이해하는 데 집중합니다.

In [ ]:
x = np.array([1.0, 2.0])

W = np.array([
    [0.5, 0.2],
    [-0.3, 0.8]
])

b = np.array([0.1, -0.1])

z = W @ x + b
h = sigmoid(z)

print("x shape :", x.shape)
print("W shape :", W.shape)
print("b shape :", b.shape)
print("z shape :", z.shape)
print("h shape :", h.shape)

print("\nz =", z)
print("h =", h)

### 기억할 것

Lecture 5의 vector-matrix backpropagation에서 notation이 복잡해져도
각 변수와 그 gradient는 대응되는 shape를 가집니다.

예:

```text
W        ↔ dL/dW
b        ↔ dL/db
z        ↔ dL/dz
h        ↔ dL/dh
```

따라서 먼저 **어떤 변수의 gradient를 계산하고 있는지**를 확인하는 습관이 중요합니다.

# Part VI. Automatic Differentiation

## 18. TensorFlow는 Backpropagation을 자동으로 계산한다

실제로 neural network를 학습할 때는 우리가 모든 derivative를 직접 계산하지 않습니다.

TensorFlow의 `GradientTape`는 forward 연산을 기록한 뒤
chain rule을 이용해 gradient를 자동으로 계산합니다.

하지만 내부적으로 수행하는 원리는 앞에서 직접 계산한 backpropagation과 같습니다.

In [ ]:
import tensorflow as tf

# 같은 scalar 2-layer network
x_tf = tf.constant(1.5, dtype=tf.float32)
y_tf = tf.constant(2.0, dtype=tf.float32)

w1_tf = tf.Variable(0.8, dtype=tf.float32)
b1_tf = tf.Variable(0.1, dtype=tf.float32)

w2_tf = tf.Variable(1.2, dtype=tf.float32)
b2_tf = tf.Variable(-0.2, dtype=tf.float32)

with tf.GradientTape() as tape:
    # Forward
    z1_tf = w1_tf * x_tf + b1_tf
    h_tf = tf.math.sigmoid(z1_tf)

    z2_tf = w2_tf * h_tf + b2_tf
    y_hat_tf = z2_tf

    loss_tf = 0.5 * (y_hat_tf - y_tf) ** 2

grads = tape.gradient(
    loss_tf,
    [w1_tf, b1_tf, w2_tf, b2_tf]
)

print("loss =", float(loss_tf))

for name, grad in zip(
    ["dL/dw1", "dL/db1", "dL/dw2", "dL/db2"],
    grads
):
    print(name, "=", float(grad))

### 확인할 내용

TensorFlow가 계산한 gradient를 앞에서 직접 계산한 값과 비교하세요.

핵심은 다음과 같습니다.

> `GradientTape`가 backpropagation을 **없애는 것**이 아니라,  
> 우리가 직접 쓰지 않아도 **자동으로 수행해 주는 것**입니다.

## 19. Keras에서는 더 많은 부분을 자동화한다

Keras의 `model.fit()`은 내부적으로 다음을 반복합니다.

```text
Forward
→ Loss
→ Backpropagation
→ Optimizer update
```

학생이 직접 derivative를 작성하지 않아도 됩니다.

In [ ]:
tf.random.set_seed(0)

# 아주 작은 regression dataset
X = np.array(
    [[0.0], [1.0], [2.0], [3.0]],
    dtype=np.float32
)

Y = np.array(
    [[0.0], [1.0], [4.0], [9.0]],
    dtype=np.float32
)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.05),
    loss="mse"
)

history = model.fit(
    X,
    Y,
    epochs=300,
    verbose=0
)

plt.plot(history.history["loss"])
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Keras Training")
plt.grid(alpha=0.3)
plt.show()

# Part VII. 왜 깊어지면 Gradient가 약해질 수 있는가?

## 20. Backpropagation은 derivative를 반복해서 곱한다

Chain rule 때문에 깊은 network에서는

$$
\frac{\partial \mathcal{L}}{\partial h^{(1)}}
=
\frac{\partial \mathcal{L}}{\partial h^{(L)}}
\cdot
\frac{\partial h^{(L)}}{\partial h^{(L-1)}}
\cdots
\frac{\partial h^{(2)}}{\partial h^{(1)}}
$$

처럼 local derivative가 반복해서 곱해집니다.

만약 매 layer에서 대략 `0.5`가 곱해진다고 생각하면

```text
1 layer : 0.5
2 layers: 0.25
3 layers: 0.125
...
```

처럼 앞쪽 layer로 갈수록 gradient가 작아질 수 있습니다.

In [ ]:
depth = np.arange(1, 21)

local_derivative = 0.5
gradient_size = local_derivative ** depth

plt.plot(depth, gradient_size, marker="o")
plt.xlabel("Number of Backpropagation Steps")
plt.ylabel("Gradient Magnitude")
plt.title("Repeated Multiplication of Small Derivatives")
plt.grid(alpha=0.3)
plt.show()

### 확인할 내용

작은 derivative가 반복해서 곱해지면 gradient가 0에 가까워집니다.

이것이 Lecture 5에서 소개하는 **vanishing gradient**의 기본 직관입니다.

지금 단계에서는 해결 방법을 외우기보다

> **깊은 network에서는 backward path도 길어진다.**

는 점을 이해하면 충분합니다.

# Part VIII. 직접 해보기

## 21. Backpropagation 체크리스트

아래 질문에 코드를 보지 않고 말로 답해 보세요.

1. Forward pass는 어느 방향으로 진행됩니까?
2. Backward pass는 어느 방향으로 진행됩니까?
3. `z`와 activation `h`는 무엇이 다릅니까?
4. gradient의 부호가 양수라는 것은 무엇을 의미합니까?
5. 왜 gradient descent에서는 gradient를 **빼는 방향**으로 움직입니까?
6. chain rule에서 `upstream gradient × local derivative`는 무엇을 의미합니까?
7. hidden layer의 gradient를 계산하려면 왜 output layer의 gradient가 먼저 필요합니까?
8. TensorFlow의 automatic differentiation은 어떤 계산을 자동화합니까?

## 22. 직접 해보기 1 — Learning Rate

앞의 manual training code에서 learning rate를 다음처럼 바꿔 보세요.

```python
lr = 0.001
lr = 0.1
lr = 1.0
```

### 관찰할 내용

- 너무 작은 learning rate: 학습이 느립니다.
- 적절한 learning rate: loss가 안정적으로 감소합니다.
- 너무 큰 learning rate: loss가 불안정하거나 증가할 수 있습니다.

## 23. 직접 해보기 2 — Activation의 영향

2-layer scalar network에서 sigmoid 대신 ReLU를 사용해 보세요.

특히 pre-activation `z1`이 음수가 되는 parameter를 선택했을 때
hidden activation과 gradient가 어떻게 되는지 관찰해 보세요.

> 이 실험은 이후 activation function과 gradient flow를 이해하는 데 연결됩니다.

## 24. 도전 — Numerical Gradient로 모든 Parameter 확인

2-layer scalar network의

- `w1`
- `b1`
- `w2`
- `b2`

각각에 대해 finite difference를 사용해 numerical gradient를 계산하고,
backpropagation 결과와 비교해 보세요.

이 테스트를 **gradient checking**이라고 합니다.

# 25. 정리

Lecture 5에서 가장 중요한 내용을 한 흐름으로 정리하면 다음과 같습니다.

### Forward Pass

$$
\mathbf{z}^{(l)}
=
\mathbf{W}^{(l)}
\mathbf{h}^{(l-1)}
+
\mathbf{b}^{(l)}
$$

$$
\mathbf{h}^{(l)}
=
\sigma(\mathbf{z}^{(l)})
$$

입력에서 prediction 방향으로 **값**을 계산합니다.

### Loss

Prediction과 ground truth의 차이를 계산합니다.

### Backward Pass

Loss에서 입력 방향으로 **gradient**를 전달합니다.

각 단계의 핵심은

$$
\text{downstream gradient}
=
\text{upstream gradient}
\times
\text{local derivative}
$$

입니다.

### Gradient Descent

계산된 gradient를 사용해 parameter를 업데이트합니다.

$$
\mathbf{W}
\leftarrow
\mathbf{W}
-
\eta
\frac{\partial \mathcal{L}}
{\partial \mathbf{W}}
$$

---

## 꼭 기억할 세 문장

1. **Forward pass는 값을 계산한다.**
2. **Backward pass는 loss에 대한 gradient를 뒤에서 앞으로 전달한다.**
3. **Backpropagation은 chain rule을 network에 반복 적용한 것이다.**

행렬과 Jacobian notation이 복잡해져도 이 세 가지 원리는 변하지 않습니다.